Imports

In [38]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn import tree

# Point to project root
project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.paths import PROCESSED_DATA_DIR
from scipy.stats import randint


Baseline random forest

In [39]:
# 1. Load the preprocessed data
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze()

# 2. Initialize the model
clf = RandomForestClassifier(random_state=42)

# 3. Train the model
clf.fit(X_train, y_train)

# 4. Make predictions on test set
y_pred = clf.predict(X_test)

# 5. Evaluate the results
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:", classification_report(y_test, y_pred))

Accuracy: 0.9640158330334653

Classification report:               precision    recall  f1-score   support

           0       0.97      0.98      0.98      4115
           1       0.94      0.93      0.93      1443

    accuracy                           0.96      5558
   macro avg       0.95      0.95      0.95      5558
weighted avg       0.96      0.96      0.96      5558



In [41]:
# 1. Initialize the baseline model
rf_model = RandomForestClassifier(random_state=42)

# 2. Define the Inner and Outer CV
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

# 3. Define the Random Forest search space
rf_space = {
    'n_estimators': randint(100, 300), 
    'max_features': ['sqrt', 'log2'],  
    'max_depth': randint(5, 20), 
    'min_samples_split': randint(2, 11), 
    'min_samples_leaf': randint(1, 11)
}

# 4. Set up the Inner Loop (The Random Search)
rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_space,
    n_iter=20,          
    scoring='accuracy',
    n_jobs=-1,         
    cv=inner_cv,
    random_state=1
)

# 5. Execute the Outer Loop (Nested CV Evaluation)
rf_nested_scores = cross_val_score(
    estimator=rf_search, 
    X=X_train, 
    y=y_train, 
    cv=outer_cv, 
    scoring='accuracy',
    n_jobs=-1
)

# 6. View the Nested CV results
print(f"\nRF Nested CV Accuracy Scores: {rf_nested_scores}")
print(f"RF Expected Generalization Accuracy: {np.mean(rf_nested_scores):.3f} +/- {np.std(rf_nested_scores):.3f}")

# 7. Extract the model
rf_search.fit(X_train, y_train)
best_rf_model = rf_search.best_estimator_

print(f"Final Best Parameters: {rf_search.best_params_}")

# 8. Evaluate
rf_final_accuracy = best_rf_model.score(X_test, y_test)
print(f"Final Test Set Accuracy: {rf_final_accuracy:.3f}")


RF Nested CV Accuracy Scores: [0.96993061 0.97184728 0.96876205 0.96876205 0.96567682]
RF Expected Generalization Accuracy: 0.969 +/- 0.002
Final Best Parameters: {'max_depth': 12, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 251}
Final Test Set Accuracy: 0.967
